# 가족 유전체(Family of Five) 데이터 전처리 노트북

이 노트북은 23andMe raw genotype 파일 5개(Father / Mother / Child 1 / Child 2 / Child 3)를 대상으로 다음 작업을 순서대로 진행합니다.

1. 라이브러리 준비 및 파일 업로드
2. 원자료 로드 함수 정의 (주석 헤더 처리)
3. **데이터사전** 작성 — SNP 총개수, 염색체별 분포, 유전자형 구성(동형/이형/반접합), 결측(no-call) 비율, 성별 추정
4. **전처리** — 타입 변환, 결측/노콜 플래그, 이형접합 파생 컬럼, 중복 제거
5. **rsid 기준 5개 파일 병합** — 가족 구성원 간 비교를 위한 핵심 단계
6. 결과 저장 (데이터사전 CSV, 전처리 로그, 병합 테이블)

> Google Colab에서 위에서 아래로 순서대로 셀을 실행하면 됩니다. 각 셀 위 설명(마크다운)을 먼저 읽고 실행해 주세요.

## 1단계. 라이브러리 준비

이 노트북은 `pandas`, `numpy`만 사용합니다. Google Colab에는 기본으로 설치되어 있어 별도 설치가 필요 없습니다.

In [ ]:
import pandas as pd
import numpy as np
from functools import reduce
import os, re

pd.set_option('display.width', 200)
print("pandas 버전:", pd.__version__)

## 2단계. 데이터 파일 업로드

23andMe raw genotype CSV 파일 5개(Father, Mother, Child 1, Child 2, Child 3에 해당하는 파일)를 업로드합니다.

- Google Colab에서 실행 중이면 아래 셀을 실행했을 때 **파일 선택 버튼**이 나타납니다. 5개 파일을 한 번에 선택해 업로드하세요.
- Colab이 아닌 로컬 Jupyter 환경이라면, 업로드 대신 파일들을 이 노트북과 같은 폴더에 두고 이 셀은 건너뛰어도 됩니다(3단계에서 직접 경로를 지정).
- 파일명에 정답이 정해져 있지 않아도 됩니다. 파일명에 `father`, `mother`, `child_1`/`child1`, `child_2`/`child2`, `child_3`/`child3` 같은 키워드가 들어 있으면 3단계에서 자동으로 인식합니다. 인식이 잘못되면 3단계 코드에서 직접 매핑을 수정하세요.

In [ ]:
uploaded_paths = []

try:
    from google.colab import files
    print("Colab 환경 감지됨 — 파일 선택 창이 열립니다. 5개 CSV 파일을 모두 선택하세요.")
    uploaded = files.upload()
    uploaded_paths = list(uploaded.keys())
except ImportError:
    print("Colab이 아닌 환경입니다. 현재 폴더(.)에서 CSV 파일을 직접 찾습니다.")
    uploaded_paths = [f for f in os.listdir('.') if f.lower().endswith('.csv')]

print("\n업로드/발견된 파일:")
for f in uploaded_paths:
    print(" -", f)

## 3단계. 구성원별 파일 경로 매핑

업로드된 파일명에서 키워드를 찾아 Father/Mother/Child 1/Child 2/Child 3에 자동으로 연결합니다.
자동 매핑 결과가 화면에 출력되므로, **실행 후 반드시 매핑이 올바른지 확인**하세요. 잘못 매핑됐다면 `FILES` 딕셔너리를 직접 수정하면 됩니다.

In [ ]:
KEYWORDS = {
    'Father':  [r'father'],
    'Mother':  [r'mother'],
    'Child 1': [r'child[_\s]?1\b'],
    'Child 2': [r'child[_\s]?2\b'],
    'Child 3': [r'child[_\s]?3\b'],
}

FILES = {}
for person, patterns in KEYWORDS.items():
    match = None
    for f in uploaded_paths:
        for pat in patterns:
            if re.search(pat, f, re.IGNORECASE):
                match = f
                break
        if match:
            break
    if match:
        FILES[person] = match
    else:
        print(f"[경고] '{person}'에 해당하는 파일을 자동으로 찾지 못했습니다 — 아래에서 직접 지정해 주세요.")

print("자동 매핑 결과:")
for person, path in FILES.items():
    print(f"  {person:8s} -> {path}")

# 자동 매핑이 틀렸다면 아래처럼 직접 덮어쓰세요 (예시, 필요할 때만 주석 해제):
# FILES['Father']  = '실제_파일명.csv'
# FILES['Mother']  = '실제_파일명.csv'
# FILES['Child 1'] = '실제_파일명.csv'
# FILES['Child 2'] = '실제_파일명.csv'
# FILES['Child 3'] = '실제_파일명.csv'

assert len(FILES) == 5, "5개 파일이 모두 매핑되어야 다음 단계로 진행할 수 있습니다."

## 4단계. 원자료 로드 함수

23andMe raw 파일은 파일 상단에 `#`으로 시작하는 주석(설명) 줄이 여러 줄 있고, 그 아래부터 실제 데이터가 옵니다.
데이터는 콤마로 구분되어 있으며 컬럼은 `rsid`(SNP 식별자), `chromosome`(염색체), `position`(염색체 상 위치), `genotype`(유전자형, 예: AG) 4개입니다.

In [ ]:
def load_raw(path):
    """23andMe raw genotype 파일을 로드. 주석 헤더(#)는 건너뛰고 4개 컬럼만 사용."""
    df = pd.read_csv(path, comment='#', names=['rsid', 'chromosome', 'position', 'genotype'], low_memory=False)
    df['chromosome'] = df['chromosome'].astype(str)
    return df

raw = {name: load_raw(path) for name, path in FILES.items()}

for name, df in raw.items():
    print(f"{name:8s}: {len(df):>7,} 행")

## 5단계. 데이터사전(Data Dictionary) 작성

각 구성원 파일에 대해 다음을 계산합니다.

- SNP 총개수, 염색체(1-22, X, Y, MT)별 분포
- 유전자형 구성: 동형접합(homozygous, 예 AA) / 이형접합(heterozygous, 예 AG) / 반접합(hemizygous, X/Y/MT의 단일 문자 콜) 비율
- 결측치(no-call, `--` 등)와 삽입/결실(indel) 비율
- 성별 추정: 23andMe 칩은 생물학적 성별과 무관하게 Y염색체 프로브 위치를 항상 포함하므로, 단순히 Y행이 있는지가 아니라 그 위치에서 **유효 판독(no-call이 아닌) 비율**로 남녀를 구분합니다.

In [ ]:
profile_rows = []
chrom_dist = {}

for name, df in raw.items():
    n_total = len(df)
    gt = df['genotype'].astype(str)

    is_nocall    = gt.isin(['--', '00', 'DI', 'ID'])
    is_indel     = gt.str.contains('I|D', regex=True) & ~is_nocall
    is_biallelic = gt.str.match(r'^[ACGT]{2}$')
    is_hemi      = gt.str.match(r'^[ACGT]$')          # X/Y/MT 등 단일 문자 유효 콜
    is_homo      = is_biallelic & (gt.str[0] == gt.str[1])
    is_het       = is_biallelic & (gt.str[0] != gt.str[1])

    y_mask       = df['chromosome'] == 'Y'
    n_y_rows     = int(y_mask.sum())                   # 칩에 포함된 Y 프로브 자리 수(성별 무관하게 항상 존재)
    n_y_valid    = int((y_mask & ~is_nocall).sum())     # 그중 실제로 유효 판독된 콜 수
    pct_y_valid  = round(n_y_valid / n_y_rows * 100, 1) if n_y_rows else 0.0

    profile_rows.append({
        'person': name,
        'n_snp_total': n_total,
        'n_chrom_1_22': df['chromosome'].isin([str(i) for i in range(1, 23)]).sum(),
        'n_chrom_X': int((df['chromosome'] == 'X').sum()),
        'n_chrom_Y_rows': n_y_rows,
        'n_chrom_Y_valid_calls': n_y_valid,
        'pct_chrom_Y_valid': pct_y_valid,
        'n_chrom_MT': int((df['chromosome'] == 'MT').sum()),
        'n_homozygous': int(is_homo.sum()),
        'pct_homozygous': round(is_homo.mean() * 100, 2),
        'n_heterozygous': int(is_het.sum()),
        'pct_heterozygous': round(is_het.mean() * 100, 2),
        'n_hemizygous': int(is_hemi.sum()),
        'pct_hemizygous': round(is_hemi.mean() * 100, 2),
        'n_indel': int(is_indel.sum()),
        'pct_indel': round(is_indel.mean() * 100, 2),
        'n_nocall': int(is_nocall.sum()),
        'pct_nocall': round(is_nocall.mean() * 100, 2),
        # Y 유효판독 비율이 20%를 넘으면 남성으로, 거의 0%면 여성으로 추정
        'sex_inferred': f'남성 (Y 유효판독 {pct_y_valid}%)' if pct_y_valid > 20 else '여성 (Y 전부 no-call)',
    })

    chrom_dist[name] = df['chromosome'].value_counts()

data_dictionary = pd.DataFrame(profile_rows)
data_dictionary

In [ ]:
# 염색체별 SNP 개수 표 (1~22, X, Y, MT 순, 구성원별 컬럼)
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT']
chrom_distribution = pd.DataFrame(
    {name: [dist.get(c, 0) for c in chrom_order] for name, dist in chrom_dist.items()},
    index=chrom_order,
)
chrom_distribution.index.name = 'chromosome'
chrom_distribution

## 6단계. 전처리 — 타입 변환 및 플래그 컬럼 추가

각 파일에 대해 다음을 적용합니다.

1. `chromosome`을 문자열로 유지(숫자 염색체와 X/Y/MT를 함께 다루기 위함), `position`은 정수형으로 변환
2. `genotype`이 `--` 등으로 표시된 결측(no-call) 행은 **삭제하지 않고** `is_nocall` 플래그 컬럼으로 표시
3. `is_indel`, `is_biallelic`, `is_hemizygous` 플래그 컬럼 추가
4. 두 대립유전자 문자가 다른 경우를 표시하는 `is_heterozygous` 파생 컬럼 추가
5. rsid 중복 행 제거(있을 경우)

In [ ]:
log_lines = ["=== 전처리 로그 ==="]

def logp(msg):
    print(msg)
    log_lines.append(msg)

clean = {}
for name, path in FILES.items():
    df = load_raw(path)
    n0 = len(df)

    df['position'] = pd.to_numeric(df['position'], errors='coerce').astype('Int64')
    df['genotype'] = df['genotype'].astype(str)

    df['is_nocall']      = df['genotype'].isin(['--', '00', 'DI', 'ID'])
    df['is_indel']       = df['genotype'].str.contains('I|D', regex=True) & ~df['is_nocall']
    df['is_biallelic']   = df['genotype'].str.match(r'^[ACGT]{2}$')
    df['is_hemizygous']  = df['genotype'].str.match(r'^[ACGT]$')
    df['is_heterozygous'] = df['is_biallelic'] & (df['genotype'].str[0] != df['genotype'].str[1])

    n_dup = int(df['rsid'].duplicated().sum())
    if n_dup:
        df = df.drop_duplicates('rsid')
    n1 = len(df)

    logp(f"[{name}] 로드: {n0:,}행 -> 중복 rsid {n_dup:,}건 제거 -> {n1:,}행 확정 "
         f"(no-call {int(df['is_nocall'].sum()):,}건은 삭제하지 않고 플래그만 부여)")

    clean[name] = df

clean['Father'].head()

## 7단계. rsid 기준 5개 파일 병합

**가족 구성원 간 비교를 하려면 이 단계가 핵심입니다.** 5개 파일을 `rsid`를 기준으로 하나의 표로 합쳐, 같은 SNP 위치에서 다섯 명의 유전자형을 나란히 비교할 수 있게 만듭니다.

- 구성원마다 사용한 23andMe 칩 버전이 달라 rsid 구성이 완전히 같지 않으므로 **outer join(합집합)** 으로 병합합니다.
- `chromosome`/`position`은 파일마다 값이 같아야 정상이지만, 혹시 다를 경우 먼저 등장한(Father 우선) 값을 사용합니다.
- 파이썬 for-loop로 300만 개 행을 하나씩 순회하는 대신, pandas의 `merge`(outer)와 `bfill`을 사용해 벡터 연산으로 처리해 Colab에서도 빠르게 실행됩니다.

In [ ]:
# 구성원별로 chromosome/position/genotype 컬럼 이름에 구성원명을 붙여서 준비
person_frames = []
for name, df in clean.items():
    key = name.replace(' ', '')  # 'Child 1' -> 'Child1' (컬럼명에 공백 방지)
    d = df[['rsid', 'chromosome', 'position', 'genotype']].rename(columns={
        'chromosome': f'chrom_{key}',
        'position': f'pos_{key}',
        'genotype': f'gt_{key}',
    })
    person_frames.append(d)

# rsid 기준 outer 병합 (5개 파일 순차 병합)
merged_full = reduce(lambda left, right: pd.merge(left, right, on='rsid', how='outer'), person_frames)

# chromosome/position은 구성원 중 먼저 값이 있는 컬럼을 사용 (Father 우선순위)
chrom_cols = [c for c in merged_full.columns if c.startswith('chrom_')]
pos_cols   = [c for c in merged_full.columns if c.startswith('pos_')]
merged_full['chromosome'] = merged_full[chrom_cols].bfill(axis=1).iloc[:, 0]
merged_full['position']   = merged_full[pos_cols].bfill(axis=1).iloc[:, 0]
merged_full = merged_full.drop(columns=chrom_cols + pos_cols)

# 컬럼 순서 정리: rsid, chromosome, position, gt_Father, gt_Mother, gt_Child1, gt_Child2, gt_Child3
gt_cols = [c for c in merged_full.columns if c.startswith('gt_')]
merged_full = merged_full[['rsid', 'chromosome', 'position'] + gt_cols]

n_union = len(merged_full)
n_all5 = merged_full.dropna(subset=gt_cols).shape[0]

logp(f"\n5개 파일 rsid 합집합: {n_union:,}개")
logp(f"5명 전원에게서 공통으로 존재하는 rsid(완전 교집합): {n_all5:,}개")

merged_full.head()

In [ ]:
# 칩 버전 차이에 대한 참고 메모 (데이터사전에서 확인한 내용을 로그에 남김)
logp("\n[참고] Child 2/Child 3는 Father/Mother/Child 1과 다른(더 신형) 23andMe 칩을 사용한 것으로 보입니다 "
     "(SNP 총개수·이형접합 비율이 두 그룹 간에 체계적으로 다름 -> 생물학적 차이가 아니라 칩 설계 차이일 가능성이 높으므로, "
     "구성원 간 이형접합도 등을 비교할 때는 이 점을 감안해야 합니다).")
print("완료")

## 8단계. 결과 저장

아래 3개 파일을 만듭니다.

- `data_dictionary.csv` — 구성원별 데이터사전 요약표
- `chrom_distribution.csv` — 염색체별 SNP 개수 표
- `merged_family_genome.csv.gz` — rsid 기준으로 병합된 5인 유전형 테이블 (전처리·인사이트 단계에서 재사용)
- `preprocess_log.txt` — 전처리 과정 로그

In [ ]:
data_dictionary.to_csv('data_dictionary.csv', index=False, encoding='utf-8-sig')
chrom_distribution.to_csv('chrom_distribution.csv', encoding='utf-8-sig')
merged_full.to_csv('merged_family_genome.csv.gz', index=False, compression='gzip')

with open('preprocess_log.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))

print("저장 완료:")
print(" - data_dictionary.csv")
print(" - chrom_distribution.csv")
print(" - merged_family_genome.csv.gz  (", len(merged_full), "행 x", merged_full.shape[1], "열 )")
print(" - preprocess_log.txt")

## 9단계 (선택). 결과 파일 다운로드

Colab에서 실행 중이라면 아래 셀로 결과 파일을 로컬 컴퓨터에 바로 내려받을 수 있습니다.

In [ ]:
try:
    from google.colab import files
    for fname in ['data_dictionary.csv', 'chrom_distribution.csv', 'merged_family_genome.csv.gz', 'preprocess_log.txt']:
        files.download(fname)
except ImportError:
    print("Colab이 아닌 환경에서는 파일이 이미 현재 폴더에 저장되어 있습니다.")

## 정리

이 노트북에서 만든 `merged_family_genome.csv.gz`(rsid 기준 5인 병합 테이블)와 `data_dictionary.csv`는 이후 단계인 **시각화**(염색체별 SNP 분포, 이형접합 비율, 가족 간 일치율 히트맵, 덴드로그램 등)와 **인사이트 분석**(멘델 유전 법칙 검증, 부모 기원 추정, 형제자매 유사도 비교 등)의 입력 데이터로 그대로 이어서 사용할 수 있습니다.

> 참고: 이 데이터셋은 미성년 자녀가 포함된 가족 유전체이므로, 질병 위험·비만·인지능력 등 예측적 함의를 가지는 SNP은 이후 분석에서도 다루지 않는 것을 권장합니다.